# 03. ComprasNet 원본 구조·품질 진단

세 원본 CSV를 수정하지 않고 구조와 품질만 점검한다. `Participantes.csv`는 약 2GB이므로
표본과 청크를 사용한다. 이 노트북은 processed 파일을 만들지 않으며, 표본 기반 결과에는
반드시 그 범위를 표시한다.

In [1]:
# 프로젝트 루트를 찾아 상대경로와 로컬 모듈 import를 일관되게 사용한다.
from pathlib import Path
import sys

def find_project_root(start: Path) -> Path:
    for candidate in [start.resolve(), *start.resolve().parents]:
        if (candidate / "src").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise RuntimeError("프로젝트 루트를 찾을 수 없습니다. 저장소 안에서 노트북을 실행하세요.")

PROJECT_ROOT = find_project_root(Path.cwd())
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))
SEED = 20260826
PROJECT_ROOT

WindowsPath('D:/CODE/proj2/5pl')

In [2]:
# 원본 경로와 읽기 규칙을 고정하고 누락 파일은 임의 데이터 없이 중단한다.
import csv
import os
import pandas as pd
import numpy as np
from IPython.display import display

RAW_DIR = PROJECT_ROOT / "data" / "raw" / "comprasnet"
RAW_PATHS = {
    "auctions": RAW_DIR / "Licitações.csv",
    "items": RAW_DIR / "Itens.csv",
    "participants": RAW_DIR / "Participantes.csv",
}
missing = [str(path.relative_to(PROJECT_ROOT)) for path in RAW_PATHS.values() if not path.exists()]
if missing:
    raise FileNotFoundError("필수 ComprasNet 원본 파일이 없습니다: " + ", ".join(missing))
READ_OPTIONS = {"sep": ";", "encoding": "cp1252", "decimal": ",", "dtype": str}
pd.DataFrame([
    {"dataset": name, "path": str(path.relative_to(PROJECT_ROOT)), "size_gib": path.stat().st_size / 2**30}
    for name, path in RAW_PATHS.items()
])

,dataset,path,size_gib
0,auctions,data\raw\comprasnet\Licitações.csv,0.072333
1,items,data\raw\comprasnet\Itens.csv,0.289870
2,participants,data\raw\comprasnet\Participantes.csv,1.928353


In [3]:
# cp1252 디코딩과 세미콜론 구분자 여부를 첫 줄에서 확인한다.
format_checks = []
for name, path in RAW_PATHS.items():
    first_line = path.open("r", encoding="cp1252", newline="").readline()
    columns = next(csv.reader([first_line], delimiter=";"))
    format_checks.append({
        "dataset": name,
        "cp1252_decode": True,
        "semicolon_count": first_line.count(";"),
        "column_count": len(columns),
        "first_columns": columns[:4],
    })
pd.DataFrame(format_checks)

,dataset,cp1252_decode,semicolon_count,column_count,first_columns
0,auctions,True,16,17,"[Número Licitação, Código UG, Nome UG, Código ..."
1,items,True,13,14,"[Número Licitação, Código UG, Nome UG, Código ..."
2,participants,True,12,13,"[Número Licitação, Código UG, Nome UG, Código ..."


In [4]:
# 작은 표본만 읽어 컬럼, 표본 행, dtype, 결측과 중복을 진단한다.
SAMPLE_ROWS = {"auctions": 50_000, "items": 100_000, "participants": 100_000}
samples = {
    name: pd.read_csv(path, nrows=SAMPLE_ROWS[name], low_memory=False, **READ_OPTIONS)
    for name, path in RAW_PATHS.items()
}
for name, frame in samples.items():
    print(f"\n[{name}] 표본 {len(frame):,}행 / {len(frame.columns)}열")
    display(pd.DataFrame({"column": frame.columns, "dtype": frame.dtypes.astype(str).values,
                          "missing_rate": frame.isna().mean().values}))
    display(frame.head(3))
    print("표본 완전중복 행:", int(frame.duplicated().sum()))


[auctions] 표본 50,000행 / 17열


,column,dtype,missing_rate
0,Número Licitação,object,0.00000
1,Código UG,object,0.00000
2,Nome UG,object,0.00000
3,Código Modalidade Compra,object,0.00000
4,Modalidade Compra,object,0.00000
5,Número Processo,object,0.00004
6,Objeto,object,0.00000
7,Situação Licitação,object,0.00000
8,Código Órgão Superior,object,0.00000
9,Nome Órgão Superior,object,0.00000


,Número Licitação,Código UG,Nome UG,Código Modalidade Compra,Modalidade Compra,Número Processo,Objeto,Situação Licitação,Código Órgão Superior,Nome Órgão Superior,Código Órgão,Nome Órgão,UF,Município,Data Resultado Compra,Data Abertura,Valor Licitação
0,12012,170120,DELEGACIA DA RFB EM CAMPOS GOYTACAZES,5,Pregão,15528000001201224,Objeto: Pregão Eletrônico - Contratação de emp...,Publicado,25000,Ministério da Fazenda,25000,Ministério da Fazenda - Unidades com víncul,RJ,CAMPOS DOS GOYTACAZES,02/01/2013,27/12/2012,"28296,0000"
1,12012,510918,GERENCIA EXECUTIVA PASSO FUNDO,5,Pregão,35274000865201282,Objeto: Pregão Eletrônico - Aquisição de órtes...,Evento de Alteração Publicad,33000,Ministério da Previdência Social,37202,Instituto Nacional do Seguro Social,RS,PASSO FUNDO,10/01/2013,14/12/2012,"166738,0000"
2,12012,580025,SUPERINT.FED.DE PESCA E AQUICULTURA/PB,5,Pregão,00365001949201230,Objeto: Pregão Eletrônico - Prestação de servi...,Evento de Retificação Divulg,58000,Ministério da Pesca e Aquicultura,58000,Ministério da Pesca e Aquicultura - Unidades,PB,CABEDELO,11/01/2013,19/12/2012,"60597,6000"


표본 완전중복 행: 0

[items] 표본 100,000행 / 14열


,column,dtype,missing_rate
0,Número Licitação,object,0.00000
1,Código UG,object,0.00000
2,Nome UG,object,0.00000
3,Código Modalidade Compra,object,0.00000
4,Modalidade Compra,object,0.00000
5,Número Processo,object,0.00000
6,Código Órgão,object,0.00000
7,Nome Órgão,object,0.00000
8,Código Item Compra,object,0.00963
9,Descrição,object,0.00000


,Número Licitação,Código UG,Nome UG,Código Modalidade Compra,Modalidade Compra,Número Processo,Código Órgão,Nome Órgão,Código Item Compra,Descrição,Quantidade Item,Valor Item,Código Vencedor,Nome Vencedor
0,12012,170120,DELEGACIA DA RFB EM CAMPOS GOYTACAZES,5,Pregão,15528000001201224,25000,Ministério da Fazenda - Unidades com víncul,1701200500001201200001,"INSTALACAO / MANUTENCAO - ELEVADORES, ESCADAS ...",1,"28296,0000",05379701000105,EGS ELEVADORES LTDA
1,12012,510918,GERENCIA EXECUTIVA PASSO FUNDO,5,Pregão,35274000865201282,37202,Instituto Nacional do Seguro Social,5109180500001201200001,PRÓTESE MODULAR DESARTICULAÇÃO JOELHO.,1,"8930,0000",09232222000104,CLINICA DE REABILITACAO OTTOBOCK PORTO ALEGRE ...
2,12012,510918,GERENCIA EXECUTIVA PASSO FUNDO,5,Pregão,35274000865201282,37202,Instituto Nacional do Seguro Social,5109180500001201200002,PRÓTESE MODULAR AMPUTAÇÃO TRANSTIBIAL.,1,"6279,0000",09232222000104,CLINICA DE REABILITACAO OTTOBOCK PORTO ALEGRE ...


표본 완전중복 행: 91

[participants] 표본 100,000행 / 13열


,column,dtype,missing_rate
0,Número Licitação,object,0.0
1,Código UG,object,0.0
2,Nome UG,object,0.0
3,Código Modalidade Compra,object,0.0
4,Modalidade Compra,object,0.0
5,Número Processo,object,0.0
6,Código Órgão,object,0.0
7,Nome Órgão,object,0.0
8,Código Item Compra,object,0.0
9,Descrição Item Compra,object,0.0


,Número Licitação,Código UG,Nome UG,Código Modalidade Compra,Modalidade Compra,Número Processo,Código Órgão,Nome Órgão,Código Item Compra,Descrição Item Compra,Código Participante,Nome Participante,Flag Vencedor
0,12012,170120,DELEGACIA DA RFB EM CAMPOS GOYTACAZES,5,Pregão,15528000001201224,25000,Ministério da Fazenda - Unidades com víncul,1701200500001201200001,"INSTALACAO / MANUTENCAO - ELEVADORES, ESCADAS ...",05379701000105,EGS ELEVADORES LTDA,SIM
1,12012,510918,GERENCIA EXECUTIVA PASSO FUNDO,5,Pregão,35274000865201282,37202,Instituto Nacional do Seguro Social,5109180500001201200001,PRÓTESE MODULAR DESARTICULAÇÃO JOELHO.,03233236000166,ROSEMBERG CARRIEL VIANA,NÃO
2,12012,510918,GERENCIA EXECUTIVA PASSO FUNDO,5,Pregão,35274000865201282,37202,Instituto Nacional do Seguro Social,5109180500001201200001,PRÓTESE MODULAR DESARTICULAÇÃO JOELHO.,87013710000134,ORTOCOM ORTOPEDIA E COM DE APARELHOS ORTOPEDIC...,NÃO


표본 완전중복 행: 1892


In [5]:
# 날짜·상태·수량·금액 이상치를 표본 범위에서 확인한다.
def brazil_number(series: pd.Series) -> pd.Series:
    return pd.to_numeric(series.astype("string").str.replace(".", "", regex=False).str.replace(",", ".", regex=False), errors="coerce")

auctions = samples["auctions"]
items = samples["items"]
date_summary = []
for column in ["Data Resultado Compra", "Data Abertura"]:
    parsed = pd.to_datetime(auctions[column], dayfirst=True, errors="coerce")
    date_summary.append({"column": column, "min": parsed.min(), "max": parsed.max(), "invalid_rate": parsed.isna().mean()})
display(pd.DataFrame(date_summary))
display(auctions["Situação Licitação"].value_counts(dropna=False).head(20).rename("표본 건수"))
quantity = brazil_number(items["Quantidade Item"])
value = brazil_number(items["Valor Item"])
pd.Series({
    "item_sample_rows": len(items), "quantity_le_zero": int((quantity <= 0).sum()),
    "value_le_zero": int((value <= 0).sum()), "quantity_parse_fail": int(quantity.isna().sum()),
    "value_parse_fail": int(value.isna().sum()),
})

,column,min,max,invalid_rate
0,Data Resultado Compra,2013-01-02,2015-02-26,0.00000
1,Data Abertura,2011-11-21,2015-02-19,0.00036


Situação Licitação
Evento de Resultado de Julgame    22553
Publicado                         21805
Divulgado                          1799
Evento de Alteração Publicad       1227
Evento de Suspensão Publicado       858
Evento de Adiamento Publicado       631
Evento de Revogação Publicad        308
Evento de Alteração de Resul        209
Evento de Retificação Divulg        132
Evento de Alteração Divulgad        124
Evento de Adiamento Divulgado       116
Evento de Reabertura de Prazo        90
Evento de Anulação Publicado         74
Evento de Retificação Public         29
Evento de Revogação Divulgad         27
Inválido                             10
Evento de Anulação Divulgado          5
Evento de Habilitação Public          3
Name: 표본 건수, dtype: int64

item_sample_rows       100000
quantity_le_zero            0
value_le_zero               0
quantity_parse_fail         0
value_parse_fail            0
dtype: int64

In [6]:
# 식별코드 형식과 표본 키 결합 가능성을 확인한다.
def digit_profile(series: pd.Series) -> dict:
    values = series.dropna().astype("string")
    lengths = values.str.len()
    return {"non_null": len(values), "digit_only_rate": values.str.fullmatch(r"\d+").mean(),
            "min_length": lengths.min(), "max_length": lengths.max(), "leading_zero_rate": values.str.startswith("0").mean()}

id_checks = pd.DataFrame([
    {"field": "item_code_items", **digit_profile(items["Código Item Compra"])},
    {"field": "winner_cnpj", **digit_profile(items["Código Vencedor"])},
    {"field": "item_code_participants", **digit_profile(samples["participants"]["Código Item Compra"])},
    {"field": "participant_cnpj", **digit_profile(samples["participants"]["Código Participante"])},
])
display(id_checks)
item_keys = set(items["Código Item Compra"].dropna())
participant_keys = samples["participants"]["Código Item Compra"].dropna()
auction_key_cols = ["Número Licitação", "Código UG", "Código Modalidade Compra", "Número Processo"]
auction_keys = set(map(tuple, auctions[auction_key_cols].fillna("").astype(str).to_numpy()))
item_auction_keys = list(map(tuple, items[auction_key_cols].fillna("").astype(str).to_numpy()))
pd.Series({
    "participant_item_join_rate_sample": participant_keys.isin(item_keys).mean(),
    "item_auction_composite_join_rate_sample": np.mean([key in auction_keys for key in item_auction_keys]),
})

,field,non_null,digit_only_rate,min_length,max_length,leading_zero_rate
0,item_code_items,99037,1.00000,21,22,0.00000
1,winner_cnpj,100000,0.99985,11,14,0.50071
2,item_code_participants,100000,1.00000,1,22,0.01997
3,participant_cnpj,100000,0.98000,2,14,0.50264


participant_item_join_rate_sample          0.93012
item_auction_composite_join_rate_sample    1.00000
dtype: float64

In [7]:
# 설명 키워드로 상품·서비스 혼재 정도를 탐색한다(공식 분류가 아닌 대리변수).
description = items["Descrição"].fillna("").str.upper()
service_pattern = r"SERVI|MANUTEN|INSTALA|LOCA|CONTRATA|CONSULT|TREINAMENTO"
service_proxy = description.str.contains(service_pattern, regex=True)
pd.Series({"service_keyword_proxy_rate": service_proxy.mean(),
           "goods_or_unclassified_proxy_rate": (~service_proxy).mean(),
           "note": "키워드 기반 대리분류이며 공식 상품/서비스 구분이 아님"})

service_keyword_proxy_rate                                 0.10185
goods_or_unclassified_proxy_rate                           0.89815
note                                키워드 기반 대리분류이며 공식 상품/서비스 구분이 아님
dtype: object

In [8]:
# 참여자 파일을 메모리에 올리지 않고 제한된 청크 수로 추가 검사한다.
PARTICIPANT_CHUNKS_TO_SCAN = 4
CHUNK_SIZE = 250_000
usecols = ["Código Item Compra", "Código Participante", "Flag Vencedor"]
chunk_stats = []
for chunk_number, chunk in enumerate(pd.read_csv(RAW_PATHS["participants"], usecols=usecols,
                                                   chunksize=CHUNK_SIZE, low_memory=False, **READ_OPTIONS), start=1):
    chunk_stats.append({
        "chunk": chunk_number, "rows": len(chunk), "missing_supplier_rate": chunk["Código Participante"].isna().mean(),
        "duplicate_rate_within_chunk": chunk.duplicated().mean(),
        "winner_rate": chunk["Flag Vencedor"].astype("string").str.upper().eq("SIM").mean(),
    })
    if chunk_number >= PARTICIPANT_CHUNKS_TO_SCAN:
        break
print(f"참여자 파일 앞쪽 {sum(x['rows'] for x in chunk_stats):,}행만 청크 진단했습니다. 전체 통계가 아닙니다.")
pd.DataFrame(chunk_stats)

참여자 파일 앞쪽 1,000,000행만 청크 진단했습니다. 전체 통계가 아닙니다.


,chunk,rows,missing_supplier_rate,duplicate_rate_within_chunk,winner_rate
0,1,250000,0.0,0.017752,0.119916
1,2,250000,0.0,0.004384,0.131632
2,3,250000,0.0,0.005232,0.128460
3,4,250000,0.0,0.002888,0.140188


## 진단 범위의 한계

이 노트북의 중복률, 결측률, 결합률, 상품/서비스 비율은 명시된 표본 범위의 진단치다.
전체 참여자 파일을 한 번에 적재하지 않았다. 전체 행 처리는 04에서 청크 단위로 수행하며,
원본 파일은 읽기만 한다.